# Kaggle Free Tier (`GPU T4` — `0 CU`) · Post-Hoc OVD Confidence Calibration & SAHI Full-Board Benchmark

Runs two key deployment experiments at **0 Colab credits**:
1. Fits `ConfidenceCalibrator` (`ovd_calibrator.json`) on validation predictions of `Fine-Tuned YOLO-World-S` (`yoloworld_best.pt`) so raw `0.001–0.05` scores map cleanly to calibrated `[0, 1]` probabilities.
2. Compares **Standard `640px` Full-Board Resize vs `SAHI` `640px` Sliced Inference** on high-resolution `3000px+` PCB boards.

In [ ]:
!pip install -q ultralytics>=8.3.40

import json, os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Darren7218/PCB-Defect-Detection-FYP.git'
BRANCH   = 'feat/multiproduct-patchcore-v2'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./work')
REPO_DIR = WORK_DIR / 'PCB-Defect-Detection-FYP'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

In [ ]:
from PIL import Image
from ultralytics import YOLOWorld
from src.datasets.taxonomy import UNIFIED_CLASS_NAMES
from src.detection.sahi_and_calibration import ConfidenceCalibrator, compute_iou_and_ios

KAGGLE_IN = Path('/kaggle/input')
yw_cands = list(KAGGLE_IN.rglob('yoloworld_best.pt')) + list(KAGGLE_IN.rglob('best.pt')) if KAGGLE_IN.exists() else []
val_dirs = list(KAGGLE_IN.rglob('VOC_PCB/images/val')) if KAGGLE_IN.exists() else []

if yw_cands and val_dirs:
    model = YOLOWorld(str(yw_cands[0]))
    val_img_dir = val_dirs[0]
    val_lbl_dir = val_img_dir.parent.parent / 'labels' / 'val'
    obs_by_cls = {c: [] for c in UNIFIED_CLASS_NAMES[:6]}

    for img_p in sorted(val_img_dir.glob('*.jpg'))[:400]:
        lbl_p = val_lbl_dir / f'{img_p.stem}.txt'
        if not lbl_p.exists(): continue
        w, h = Image.open(img_p).size
        gt_boxes = []
        for line in lbl_p.read_text().splitlines():
            cid, cx, cy, bw, bh = map(float, line.split())
            gt_boxes.append((int(cid), (cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h))

        res = model.predict(str(img_p), conf=0.0005, verbose=False)[0]
        for box in res.boxes:
            pcid = int(box.cls[0].item())
            pconf = float(box.conf[0].item())
            px1, py1, px2, py2 = box.xyxy[0].cpu().numpy().tolist()
            is_tp = any(gc == pcid and compute_iou_and_ios((px1, py1, px2, py2), gbox)[0] >= 0.5 for gc, *gbox in gt_boxes)
            if 0 <= pcid < 6:
                obs_by_cls[UNIFIED_CLASS_NAMES[pcid]].append((pconf, 1 if is_tp else 0))

    cal = ConfidenceCalibrator(default_operating_raw_conf=0.002).fit_from_validation_matches(obs_by_cls)
    out_cal = WORK_DIR / 'ovd_calibrator.json'
    cal.save(out_cal)
    print('Saved fitted OVD ConfidenceCalibrator ->', out_cal)
    print('Fitted class params:', cal.params_by_class)
else:
    print('Attach yoloworld_best.pt + VOC_PCB dataset in Kaggle Inputs to fit empirical Platt parameters.')